# T03: Fold-jako ja EDA

Tämä notebook ajaa vain koodia, jonka logiikka on `src/data.py`:ssä (`load_metadata`, `check_dataset`, `split_folds`, `split`). Sisältö:

1. Datan lataus ja tarkistus (T02)
2. Luokkajakauma kokonaisuutena ja foldeittain
3. Fold-jako: testi-, validointi- ja opetusfoldit jokaiselle ajolle k = 1–5
4. Äänen voimakkuus ja hiljaisuuden osuus luokittain
5. Esimerkkispektrogrammit (yksi per luokka)
6. Havainto: samasta lähdeäänitteestä tulevat leikkeet eri foldeissa

Spektrogrammien parametrit ovat tässä vain katselua varten. Lopullinen `log_mel` tehdään T04:ssä.

In [ ]:
import sys
from pathlib import Path

# Repon juuri polkuun, jotta `src` löytyy (notebook ajetaan kansiosta notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

from src.data import (ANIMAL_CLASSES, N_FOLDS, check_dataset, download_esc50,
                      load_metadata, split, split_folds)

# Colabissa: DATA_DIR = "/content/drive/MyDrive/Elainaanet-CNN/data"
DATA_DIR = ROOT / "data"
SEED = 42
np.random.seed(SEED)

## 1. Lataus ja tarkistus

In [ ]:
download_esc50(DATA_DIR)      # ei lataa uudelleen, jos data on jo paikallaan
df = load_metadata(DATA_DIR)
check_dataset(df)             # 400 leikettä, 40/luokka, 8/luokka/foldi, 5 s, 44,1 kHz, mono
print(len(df), "leikettä")
df.head()

## 2. Luokkajakauma

In [ ]:
counts = pd.crosstab(df["category"], df["fold"], margins=True, margins_name="yht.")
counts = counts.loc[ANIMAL_CLASSES + ["yht."]]
counts.columns = [f"foldi {c}" if c != "yht." else c for c in counts.columns]
counts

In [ ]:
per_class = df["target"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(ANIMAL_CLASSES, per_class.values, color="#3b6ea5", width=0.6)
ax.set_ylabel("leikkeitä")
ax.set_title("Leikkeitä per luokka (kaikki foldit)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

Luokat ovat täysin tasapainossa (40 per luokka, 8 per luokka jokaisessa foldissa), joten tarkkuus ja makro-F1 ovat vertailukelpoisia eikä luokkapainoja tarvita.

## 3. Fold-jako

`split(df, k)` palauttaa `(train_df, val_df, test_df)`. Testifoldi on k, validointi seuraava foldi kiertäen (k = 5 → 1) ja loput kolme foldia ovat opetusta. Jokainen foldi on näin kerran testinä ja kerran validointina. Kaikki mallit käyttävät tätä samaa funktiota.

In [ ]:
rows = []
for k in range(1, N_FOLDS + 1):
    train_df, val_df, test_df = split(df, k)
    train_folds, val_fold, test_fold = split_folds(k)
    rows.append({
        "ajo k": k,
        "opetusfoldit": train_folds, "validointifoldi": val_fold, "testifoldi": test_fold,
        "opetus": len(train_df), "validointi": len(val_df), "testi": len(test_df),
        "opetus/luokka": train_df["target"].value_counts().unique().tolist(),
        "testi/luokka": test_df["target"].value_counts().unique().tolist(),
    })
pd.DataFrame(rows).set_index("ajo k")

## 4. Voimakkuus ja hiljaisuus

Monet ESC-50:n leikkeet on täytetty hiljaisuudella 5 sekuntiin. Lasketaan jokaisesta leikkeestä RMS-energia kehyksittäin ja niiden kehysten osuus, jotka ovat yli 40 dB leikkeen huippua hiljaisempia. Tämä kertoo, kuinka paljon spektrogrammeissa on tyhjää ja tukee T05:n aikasiirto-augmentaation perustelua.

In [ ]:
def silence_ratio(path, top_db=40):
    y, sr = librosa.load(path, sr=None, mono=True)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    db = librosa.amplitude_to_db(rms, ref=np.max)
    return float(np.mean(db < -top_db))

df["silence"] = [silence_ratio(p) for p in df["path"]]
stats = df.groupby("category")["silence"].describe()[["mean", "50%", "min", "max"]].loc[ANIMAL_CLASSES]
stats.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
data = [df.loc[df["category"] == c, "silence"] for c in ANIMAL_CLASSES]
ax.boxplot(data, tick_labels=ANIMAL_CLASSES, widths=0.5,
           medianprops={"color": "#1f3b5c", "linewidth": 2})
ax.set_ylabel("hiljaisten kehysten osuus")
ax.set_title("Hiljaisuuden osuus leikkeessä luokittain (< -40 dB huipusta)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

Kukko-, koira- ja kissaleikkeissä on eniten hiljaisuutta (kukolla mediaani yli puolet kehyksistä), sammakko- ja lammasleikkeet ovat lähes yhtäjaksoista ääntä. Luokkien välillä on siis eroa siinä, missä kohtaa ja kuinka pitkään ääni kuuluu. Aikasiirto (T05) estää mallia oppimasta äänen sijaintia leikkeessä.

## 5. Esimerkkispektrogrammit

Ensimmäinen leike kustakin luokasta (metadatan järjestyksessä, ei valittu käsin). Log-mel, 128 mel-kaistaa, `n_fft=2048`, `hop_length=512`.

In [ ]:
examples = df.groupby("target").first()
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=True)
for ax, (target, row) in zip(axes.flat, examples.iterrows()):
    y, sr = librosa.load(row["path"], sr=None, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="mel",
                                   ax=ax, cmap="magma", vmin=-80, vmax=0)
    ax.set_xticks(range(6))
    ax.set_title(f"{target} {row['category']}\n{row['filename']}", fontsize=9)
    ax.label_outer()
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.8)
plt.show()

## 6. Havainto: sama lähdeäänite useassa foldissa

ESC-50:n foldit on tehty niin, että samasta Freesound-äänitteestä (`src_file`) leikatut pätkät ovat samassa foldissa. Eläinluokissa on kuitenkin poikkeus: alla olevat äänitteet on leikattu sekä *hen*- että *rooster*-luokkaan, ja pätkät ovat eri foldeissa. Pidämme virallisen jaon (`PLAN.md`), mutta tämä on hyvä tietää: osa kana–kukko-sekaannuksista voi johtua siitä, että sama äänite on molemmissa luokissa. Huomioidaan virheanalyysissä (T19) ja raportissa.

In [ ]:
multi = df.groupby("src_file")["fold"].nunique()
cross = df[df["src_file"].isin(multi[multi > 1].index)]
cross[["filename", "fold", "category", "src_file", "take"]].sort_values(["src_file", "fold"])

## Yhteenveto

In [ ]:
print(f"Leikkeitä: {len(df)}, luokkia: {df['target'].nunique()}, foldeja: {df['fold'].nunique()}")
print("Jako per ajo: opetus 240, validointi 80, testi 80 (24 / 8 / 8 per luokka)")
print("Hiljaisuuden keskimääräinen osuus luokittain (suurimmasta pienimpään):")
print(df.groupby("category")["silence"].mean().sort_values(ascending=False).round(2).to_string())
print(f"Eri foldeissa olevia lähdeäänitteitä: {(multi > 1).sum()} ({len(cross)} leikettä)")